# AGC 2027 | Katongan — Integrasi Geografi A × Geografi B
## Revisi klimatologi, SPI/CDD, model tandon YAS, dan peta potensi RWH per bangunan

**Input:** ZIP ekspor GEE AOI Katongan yang mencakup 3 CSV harian, 1 CSV bulanan, 1 CSV tahunan, 1 CSV atribut bangunan, dan 1 GeoJSON bangunan.

**Opsional:** CSV keluaran GEE `Katongan_RWH_PerBuilding_SpatialGEE.csv` berisi `full_plus_code`, `rain_mm_annual` untuk memakai curah hujan pada piksel CHIRPS tempat centroid bangunan berada.

**Status:** Eksperimen model berdasarkan data spasial terbuka. Bukan inventaris rumah tangga terverifikasi; bukan kelayakan air minum; bukan rekomendasi ukuran tandon aktual.

**Pustaka dasar:** Villar-Navascués et al. (2020), *Water* 12(9):2623; CHIRPS v3 Daily RNL, UCSB CHC; SPI dengan distribusi gamma per bulan kalender.

In [ ]:
# 0. Impor pustaka & folder kerja
import os, json, zipfile, io, shutil, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gamma, norm
from IPython.display import display

BASE = Path('/content/agc2027_katongan') if Path('/content').exists() else Path('/mnt/data/agc2027_katongan_run')
RAW = BASE / 'raw'
OUT = BASE / 'output'
for p in [RAW, OUT]:
    p.mkdir(parents=True, exist_ok=True)
print('Folder kerja:', BASE)

In [ ]:
# 1. Konfigurasi skenario. Semua parameter ini adalah ASUMSI.
START_YEAR, END_YEAR = 1996, 2025
ROOF_RUNOFF_COEFFICIENTS = (0.65, 0.80, 0.90)  # Rentang sensitivitas, bukan klasifikasi atap lokal
BUILDING_CONF_THRESHOLDS = (0.70, 0.75, 0.80, 0.85, 0.90)
BASE_ROOF_AREA_M2 = 100.0         # Atap contoh, bukan rata-rata rumah aktual
BASE_RUNOFF = 0.80
BASE_DEMAND_L_PER_DAY = 100.0    # Kebutuhan layanan RWH pada unit contoh, bukan standar per kapita
BASE_TANK_L = 5000.0
DRY_SEASON_MONTHS = (7, 8, 9)   # Jul-Sep berdasarkan klimatologi awal, bukan definisi kekeringan resmi
TANK_CAPACITIES_L = (1000., 3000., 5000., 10000., 20000.)
AREA_SCENARIOS_M2 = (50., 100., 150.)
DEMAND_SCENARIOS_LPD = (50., 100., 150.)
CDD_THRESHOLDS_MM = (0.5, 1.0, 2.0)
RUNOFF_ASSUMPTION = 'Runoff coefficients represent scenarios, not roof material observations.'
print('Konfigurasi berhasil dimuat.')

In [ ]:
# 2. Upload file ZIP dari Google Earth Engine
#    Dalam Colab: upload ZIP berisi 7 berkas ekspor GEE Katongan.
#    Untuk uji lokal, AGC_INPUT_ZIP_PATH boleh menunjuk path ZIP.
INPUT_ZIPS = []
# Bersihkan sisa input lama agar tidak tercampur ketika notebook dijalankan ulang.
for old in RAW.iterdir():
    if old.is_file(): old.unlink()
path_from_env = os.environ.get('AGC_INPUT_ZIP_PATH', '')
if path_from_env:
    INPUT_ZIPS = [Path(path_from_env)]
else:
    try:
        from google.colab import files
        uploaded = files.upload()
        INPUT_ZIPS = []
        for name, contents in uploaded.items():
            fpath = BASE / Path(name).name
            fpath.write_bytes(contents)
            INPUT_ZIPS.append(fpath)
    except ImportError:
        raise RuntimeError('Jalankan di Colab atau set AGC_INPUT_ZIP_PATH untuk pengujian lokal.')

assert INPUT_ZIPS, 'Tidak ada file ZIP diunggah.'
for f in INPUT_ZIPS:
    if not zipfile.is_zipfile(f):
        if f.name == 'Katongan_RWH_PerBuilding_SpatialGEE.csv':
            shutil.copyfile(f, RAW / f.name)
            continue
        raise ValueError(f'Berkas tidak dikenali: {f.name}; unggah ZIP GEE atau CSV spasial yang bernama sesuai.')
    with zipfile.ZipFile(f) as zin:
        for item in zin.infolist():
            # Ekstraksi aman: jangan ikuti path absolut / ../
            leaf = Path(item.filename).name
            if not leaf or item.is_dir():
                continue
            if leaf in {'Katongan_Buildings.geojson','Katongan_Buildings_Attributes.csv',
                        'Katongan_Annual_1996_2025.csv','Katongan_Monthly_1996_2025.csv'} \
               or (leaf.startswith('Katongan_Daily_') and leaf.endswith('.csv')) \
               or (leaf == 'Katongan_RWH_PerBuilding_SpatialGEE.csv'):
                (RAW / leaf).write_bytes(zin.read(item))
print('Berkas input yang dikenali:')
for p in sorted(RAW.iterdir()): print(' -', p.name, f'({p.stat().st_size:,} byte)')

In [ ]:
# 3. Baca data dengan validasi kolom dan tanggal yang ketat
files_daily = sorted(RAW.glob('Katongan_Daily_*.csv'))
assert len(files_daily) == 3, f'Perlu 3 CSV harian, ditemukan {len(files_daily)}.'
daily = pd.concat([pd.read_csv(f) for f in files_daily], ignore_index=True)
assert {'date','rain_mm','year','month'}.issubset(daily.columns)
daily['date'] = pd.to_datetime(daily['date'], format='%Y-%m-%d', errors='raise')
daily['rain_mm'] = pd.to_numeric(daily['rain_mm'], errors='raise')
daily['year'] = pd.to_numeric(daily['year'], errors='raise').astype(int)
daily['month'] = pd.to_numeric(daily['month'], errors='raise').astype(int)
daily = daily.sort_values('date').reset_index(drop=True)
expected_dates = pd.date_range(f'{START_YEAR}-01-01', f'{END_YEAR}-12-31', freq='D')
assert len(daily) == len(expected_dates), 'Jumlah baris hujan harian tidak cocok.'
assert daily['date'].equals(pd.Series(expected_dates, name='date')), 'Tanggal hilang, tidak urut atau duplikat.'
assert daily['rain_mm'].notna().all() and (daily['rain_mm'] >= 0).all(), 'Terdapat nilai hujan kosong atau negatif.'
assert (daily['date'].dt.year == daily['year']).all() and (daily['date'].dt.month == daily['month']).all()

annual_source = pd.read_csv(RAW / 'Katongan_Annual_1996_2025.csv')
monthly_source = pd.read_csv(RAW / 'Katongan_Monthly_1996_2025.csv')
for df, col in [(annual_source,'year'),(monthly_source,'year'),(monthly_source,'month')]:
    df[col] = pd.to_numeric(df[col],errors='raise').astype(int)
for df in (annual_source, monthly_source):
    df['rain_mm'] = pd.to_numeric(df['rain_mm'], errors='raise')

annual_calc = daily.groupby('year',as_index=False)['rain_mm'].sum()
monthly_calc = daily.groupby(['year','month'],as_index=False)['rain_mm'].sum()
annual_qc = annual_calc.merge(annual_source, on='year', suffixes=('_daily','_gee'),validate='one_to_one')
monthly_qc = monthly_calc.merge(monthly_source,on=['year','month'],suffixes=('_daily','_gee'),validate='one_to_one')
assert len(annual_qc)==30 and len(monthly_qc)==360, 'CSV agregasi tidak lengkap.'
annual_qc['abs_diff_mm']=(annual_qc.rain_mm_daily-annual_qc.rain_mm_gee).abs()
monthly_qc['abs_diff_mm']=(monthly_qc.rain_mm_daily-monthly_qc.rain_mm_gee).abs()
print('Hari lengkap:',len(daily),'tahun:',len(annual_qc),'bulan:',len(monthly_qc))
print('Selisih maksimum tahunan:',annual_qc.abs_diff_mm.max())
print('Selisih maksimum bulanan:',monthly_qc.abs_diff_mm.max())
assert annual_qc.abs_diff_mm.max() < 1e-5 and monthly_qc.abs_diff_mm.max() < 1e-5, 'QC agregasi belum lolos: hentikan sebelum analisis.'
annual_qc.to_csv(OUT/'QA_annual.csv',index=False)
monthly_qc.to_csv(OUT/'QA_monthly.csv',index=False)

In [ ]:
# 4. Statistik hujan dan klimatologi 30 tahun
annual = annual_calc.copy()
monthly = monthly_calc.copy()
climatology = monthly.groupby('month').rain_mm.agg(['mean','std','min','max']).reset_index()
climatology.columns = ['month','mean_mm','std_mm','min_mm','max_mm']
print('Hujan rata-rata tahunan:',round(annual.rain_mm.mean(),2),'mm')
print('Tahun terendah:', annual.loc[annual.rain_mm.idxmin(),['year','rain_mm']].to_dict())
print('Tahun tertinggi:', annual.loc[annual.rain_mm.idxmax(),['year','rain_mm']].to_dict())
print('Bulan rata-rata terkering:',climatology.loc[climatology.mean_mm.idxmin(),['month','mean_mm']].to_dict())
annual.to_csv(OUT/'annual_rain_1996_2025.csv',index=False)
climatology.to_csv(OUT/'monthly_climatology.csv',index=False)
monthly.to_csv(OUT/'monthly_rain.csv',index=False)
display(climatology)

In [ ]:
# 5. SPI-3 dan SPI-6: Gamma fit per bulan kalender dengan probabilitas hujan nol.
#    Semua 30 tahun dipakai sebagai periode kalibrasi; SPI pertama 2/5 bulan bernilai NaN.
#    SCIENTIFIC LIMIT: 30 sampel per bulan kalender; klasifikasi ekor ekstrem sensitif terhadap fitting.

def spi_gamma_calendar(monthly_df, window):
    a = monthly_df.sort_values(['year','month']).copy()
    a['accum_mm'] = a['rain_mm'].rolling(window=window,min_periods=window).sum()
    a['spi'] = np.nan
    for m in range(1,13):
        idx = a.index[(a.month==m) & (a.accum_mm.notna())]
        x = a.loc[idx,'accum_mm'].to_numpy(dtype=float)
        if len(x) < 20:
            warnings.warn(f'SPI-{window} bulan {m}: data kalibrasi kurang dari 20')
            continue
        positive=x[x>0]
        if len(positive) < 10:
            warnings.warn(f'SPI-{window} bulan {m}: jumlah positif terlalu sedikit')
            continue
        qzero = (x<=0).mean()
        try:
            shape,_,scale = gamma.fit(positive,floc=0)
            prob = qzero + (1-qzero)*gamma.cdf(x,shape,loc=0,scale=scale)
            prob = np.clip(prob,1e-8,1-1e-8)
            a.loc[idx,'spi'] = norm.ppf(prob)
        except (ValueError, RuntimeError, FloatingPointError) as err:
            warnings.warn(f'SPI-{window} bulan {m} gagal fit: {err}')
    return a[['year','month','accum_mm','spi']].rename(
        columns={'accum_mm':f'rain_{window}month_mm','spi':f'SPI{window}'})

spi3 = spi_gamma_calendar(monthly,3)
spi6 = spi_gamma_calendar(monthly,6)
monthly_spi = monthly.merge(spi3,on=['year','month']).merge(spi6,on=['year','month'])
monthly_spi['date'] = pd.to_datetime(dict(year=monthly_spi.year,month=monthly_spi.month,day=1))
monthly_spi.to_csv(OUT/'monthly_SPI3_SPI6.csv',index=False)
for k in ['SPI3','SPI6']:
    r=monthly_spi.loc[monthly_spi[k].idxmin()]
    print(k,'terendah:',round(r[k],3),f"({int(r.year)}-{int(r.month):02d})")

In [ ]:
# 6. CDD (consecutive dry days) per tahun; sensitif terhadap ambang CHIRPS harian.
#    Perhitungan CDD maksimum per tahun dipotong di batas 1 Januari.
def max_dry_run(flag):
    best=run=0
    for v in flag:
        run = run+1 if v else 0
        best = max(best,run)
    return best

cdd_rows=[]
for t in CDD_THRESHOLDS_MM:
    for y,g in daily.groupby('year',sort=True):
        cdd_rows.append({'year':int(y),'threshold_mm':float(t),
                         'max_CDD_days':max_dry_run(g.rain_mm.to_numpy()<t),
                         'dry_days':int((g.rain_mm<t).sum())})
cdd=pd.DataFrame(cdd_rows)
cdd.to_csv(OUT/'CDD_sensitivity.csv',index=False)
print('CDD tahunan maksimum pada threshold 1 mm:')
display(cdd[cdd.threshold_mm==1.].sort_values('max_CDD_days',ascending=False).head(8))

In [ ]:
# 7. Potensi RWH per bangunan, QC atribut.
#    Tanpa CSV sampel spasial GEE, hujan setiap bangunan memakai rerata AOI (PROXY).
#    Penggunaan footprint sebagai luas atap horizontal belum divalidasi.
b = pd.read_csv(RAW/'Katongan_Buildings_Attributes.csv')
need={'full_plus_code','confidence','area_in_meters','longitude','latitude'}
assert need.issubset(b.columns),f'Atribut bangunan kurang: {need-set(b.columns)}'
assert b.full_plus_code.notna().all() and not b.full_plus_code.duplicated().any(), 'ID bangunan hilang/duplikat'
for key in ['confidence','area_in_meters','longitude','latitude']:
    b[key]=pd.to_numeric(b[key],errors='raise')
assert b[ ['confidence','area_in_meters','longitude','latitude'] ].notna().all().all()
assert (b.area_in_meters>0).all() and b.confidence.between(0,1).all()
assert b.longitude.between(109,112).all() and b.latitude.between(-9,-6).all()
assert len(b)>0

annual_mean=float(annual.rain_mm.mean())
spatial_file=RAW/'Katongan_RWH_PerBuilding_SpatialGEE.csv'
if spatial_file.exists():
    s = pd.read_csv(spatial_file)
    assert {'full_plus_code','rain_mm_annual'}.issubset(s.columns)
    assert not s.full_plus_code.duplicated().any(), 'Ada ID duplikat pada data hujan spasial GEE'
    s['rain_mm_annual']=pd.to_numeric(s['rain_mm_annual'],errors='raise')
    b = b.merge(s[['full_plus_code','rain_mm_annual']],on='full_plus_code',how='left',validate='one_to_one')
    missing=int(b.rain_mm_annual.isna().sum())
    assert missing==0, f'Hujan per bangunan hilang untuk {missing} record; cek pengambilan sampel GEE'
    b['rain_data_method']='native_grid_pixel_at_building_centroid'
else:
    b['rain_mm_annual']=annual_mean
    b['rain_data_method']='aoi_mean_rainfall_proxy'
    print('PERHATIAN: Belum ada CSV hujan per bangunan GEE; memakai rata-rata curah hujan AOI yang sama.')

for c in ROOF_RUNOFF_COEFFICIENTS:
    b[f'RCHP_m3yr_RC{int(round(c*100))}']=(b.rain_mm_annual*b.area_in_meters*c/1000)
print('Kandidat bangunan:',len(b),'Total luas footprint (m²):',round(b.area_in_meters.sum(),2))
print('Total potensi RC=0.80 (m³/tahun):',round(b.RCHP_m3yr_RC80.sum(),2))
b.to_csv(OUT/'RWH_buildings_1996_2025_scenarios.csv',index=False)

In [ ]:
# 8. Sensitivitas jumlah bangunan, total luas, dan potensi terhadap confidence dan RC.
rows=[]
for th in BUILDING_CONF_THRESHOLDS:
    subset=b[b.confidence>=th]
    for c in ROOF_RUNOFF_COEFFICIENTS:
        rows.append({'confidence_threshold':th,'runoff_coefficient':c,
                     'building_count':len(subset),'footprint_area_m2':subset.area_in_meters.sum(),
                     'potential_m3_per_year':subset[f'RCHP_m3yr_RC{int(round(c*100))}'].sum(),
                     'rain_data_method':b.rain_data_method.iloc[0]})
conf_sensitivity=pd.DataFrame(rows)
conf_sensitivity.to_csv(OUT/'RWH_confidence_and_runoff_sensitivity.csv',index=False)
display(conf_sensitivity[conf_sensitivity.runoff_coefficient==BASE_RUNOFF])

In [ ]:
# 9. Neraca air harian YAS: yield after spillage, seluruh nilai dalam liter.
#    Urutan: hujan masuk -> air berlebih meluap -> kebutuhan dipenuhi -> sisa disimpan.
#    Tidak mengakomodasi kesesuaian kualitas air, first flush, atau losses lain.
def simulate_yas(rain_mm, roof_area_m2, runoff, demand_lpd, tank_l, initial_l=0.):
    rain = np.asarray(rain_mm,dtype=float)
    inflow = rain*roof_area_m2*runoff
    overflow = np.zeros(len(rain))
    yield_l = np.zeros(len(rain))
    storage = np.zeros(len(rain))
    deficits=np.zeros(len(rain))
    s=float(initial_l)
    for t,q in enumerate(inflow):
        before=s+q
        over=max(0.,before-tank_l)
        available=before-over
        used=min(demand_lpd,available)
        new_storage=available-used
        if not np.isclose(s+q,over+used+new_storage,atol=1e-8,rtol=1e-11):
            raise ArithmeticError(f'Mass balance error at timestep {t}')
        overflow[t],yield_l[t],storage[t]=over,used,new_storage
        deficits[t]=demand_lpd-used
        s=new_storage
    return pd.DataFrame({'inflow_L':inflow,'overflow_L':overflow,'yield_L':yield_l,
                         'storage_L':storage,'unmet_L':deficits,
                         'fully_met':yield_l>=demand_lpd-1e-7})

def summarize_model(sim, dates, demand_lpd, dry_months=DRY_SEASON_MONTHS):
    dry=np.isin(pd.DatetimeIndex(dates).month, list(dry_months))
    demand_total=demand_lpd*len(sim)
    return {'time_reliability_pct':100*sim.fully_met.mean(),
            'volume_reliability_pct':100*sim.yield_L.sum()/demand_total,
            'dry_season_time_reliability_pct':100*sim.loc[dry,'fully_met'].mean(),
            'total_inflow_m3':sim.inflow_L.sum()/1000,
            'total_spillage_m3':sim.overflow_L.sum()/1000,
            'total_delivered_m3':sim.yield_L.sum()/1000,
            'total_unmet_m3':sim.unmet_L.sum()/1000}

sim_base=simulate_yas(daily.rain_mm,BASE_ROOF_AREA_M2,BASE_RUNOFF,BASE_DEMAND_L_PER_DAY,BASE_TANK_L)
base_stats=summarize_model(sim_base,daily.date,BASE_DEMAND_L_PER_DAY)
daily_yas=pd.concat([daily[['date','year','month','rain_mm']],sim_base],axis=1)
daily_yas.to_csv(OUT/'RWH_YAS_base_daily.csv',index=False)
print('SKENARIO DASAR (bukan kondisi aktual)')
print('Area',BASE_ROOF_AREA_M2,'m², RC',BASE_RUNOFF,'demand',BASE_DEMAND_L_PER_DAY,'L/hari, tank',BASE_TANK_L,'L')
for k,v in base_stats.items(): print(k, round(v,3))

In [ ]:
# 10. Matriks skenario atap × kebutuhan × kapasitas dan sensitivitas periode kemarau.
scenarios=[]
for a in AREA_SCENARIOS_M2:
    for d in DEMAND_SCENARIOS_LPD:
        for k in TANK_CAPACITIES_L:
            sim=simulate_yas(daily.rain_mm,a,BASE_RUNOFF,d,k)
            val=summarize_model(sim,daily.date,d)
            scenarios.append({'roof_m2':a,'runoff':BASE_RUNOFF,'demand_lpd':d,'tank_l':k,**val})
scenario_table=pd.DataFrame(scenarios)
scenario_table.to_csv(OUT/'RWH_YAS_scenario_matrix.csv',index=False)
print('Jumlah kombinasi:',len(scenario_table))
display(scenario_table[(scenario_table.roof_m2==100.) &
                       (scenario_table.demand_lpd==100.)][
                       ['tank_l','time_reliability_pct','volume_reliability_pct',
                        'dry_season_time_reliability_pct']])

In [ ]:
# 11. Peta footprint berdasarkan kelas potensi; cocok untuk hasil awal WebGIS.
#     GeoJSON sumber tetap menggunakan geometri asli; RCHP berbasis skenario ditambahkan.
geo=json.loads((RAW/'Katongan_Buildings.geojson').read_text())
assert geo.get('type')=='FeatureCollection' and len(geo['features'])==len(b)
records=b.set_index('full_plus_code')
geo_codes=[f['properties']['full_plus_code'] for f in geo['features']]
assert len(set(geo_codes))==len(geo_codes),'GeoJSON mengandung ID duplikat.'
assert set(geo_codes)==set(records.index),'CSV dan GeoJSON berbeda ID; hentikan agar tidak salah join.'
for feature in geo['features']:
    x=records.loc[feature['properties']['full_plus_code']]
    feature['properties']['RCHP_m3yr_RC80']=round(float(x['RCHP_m3yr_RC80']),4)
    feature['properties']['rain_mm_annual']=round(float(x['rain_mm_annual']),3)
    feature['properties']['rain_data_method']=x['rain_data_method']
    feature['properties']['result_status']='MODELED_POTENTIAL_NOT_ACTUAL_WATER_SUPPLY'
with (OUT/'Katongan_RWH_buildings_modeled.geojson').open('w') as f:
    json.dump(geo,f,ensure_ascii=False)
print('GeoJSON RWH dibuat:',len(geo['features']),'bangunan')

In [ ]:
# 11b. Preview peta potensi RWH dari geometri footprint; bukan validasi tapak.
try:
    import geopandas as gpd
    gdf = gpd.read_file(OUT/'Katongan_RWH_buildings_modeled.geojson')
    fig,ax = plt.subplots(figsize=(9,8))
    gdf.plot(column='RCHP_m3yr_RC80', ax=ax, legend=True, linewidth=0.05)
    ax.set_title('Katongan — Potensi tangkapan teoritis per footprint (RC=0,80)')
    ax.set_xlabel('Bujur'); ax.set_ylabel('Lintang')
    ax.ticklabel_format(axis='x', style='plain', useOffset=False)
    fig.tight_layout()
    fig.savefig(OUT/'07_map_RCHP_per_building.png',dpi=180)
    plt.show()
except ImportError:
    print('Peta statis dilewati (geopandas tidak tersedia). GeoJSON tetap diekspor dan dapat dibuka di QGIS.')

In [ ]:
# 12. Grafik otomatis (tanpa mengganti angka simulasi menjadi pengukuran)
plt.figure(figsize=(11,4))
plt.plot(annual.year,annual.rain_mm,marker='o',markersize=3)
plt.title('Katongan: Curah hujan tahunan CHIRPS v3, 1996–2025')
plt.ylabel('mm/tahun');plt.xlabel('Tahun');plt.tight_layout()
plt.savefig(OUT/'01_annual_rain.png',dpi=170);plt.show()

plt.figure(figsize=(9,4))
plt.bar(climatology.month,climatology.mean_mm)
plt.xticks(range(1,13));plt.title('Katongan: Klimatologi bulanan 1996–2025')
plt.ylabel('Rata-rata mm/bulan');plt.xlabel('Bulan');plt.tight_layout()
plt.savefig(OUT/'02_monthly_climatology.png',dpi=170);plt.show()

fig,axes=plt.subplots(2,1,figsize=(12,6),sharex=True)
for ax,k in zip(axes,['SPI3','SPI6']):
    ax.plot(monthly_spi.date,monthly_spi[k]);ax.axhline(-1,ls='--',lw=1)
    ax.set_ylabel(k)
axes[0].set_title('Indeks SPI (Gamma/calendar-month), 1996–2025')
axes[1].set_xlabel('Tahun');fig.tight_layout();fig.savefig(OUT/'03_SPI3_SPI6.png',dpi=170);plt.show()

one=cdd[cdd.threshold_mm==1.]
plt.figure(figsize=(11,4));plt.bar(one.year,one.max_CDD_days)
plt.title('CDD maksimum tahunan, ambang hujan < 1 mm/hari')
plt.ylabel('Hari');plt.xlabel('Tahun');plt.tight_layout()
plt.savefig(OUT/'04_CDD_annual.png',dpi=170);plt.show()

plt.figure(figsize=(9,4));plt.hist(b.RCHP_m3yr_RC80,bins=35)
plt.title('Distribusi RCHP teoritis per kandidat bangunan (RC=0,80)')
plt.ylabel('Jumlah kandidat');plt.xlabel('m³/tahun');plt.tight_layout()
plt.savefig(OUT/'05_RCHP_per_building_hist.png',dpi=170);plt.show()

sub=scenario_table[(scenario_table.roof_m2==100.)&(scenario_table.demand_lpd==100.)]
plt.figure(figsize=(9,4))
plt.plot(sub.tank_l,sub.time_reliability_pct,marker='o',label='Sepanjang tahun')
plt.plot(sub.tank_l,sub.dry_season_time_reliability_pct,marker='s',label='Juli–September')
plt.title('Sensitivitas keandalan terhadap kapasitas tandon (skenario)')
plt.xlabel('Kapasitas tandon (L)');plt.ylabel('Hari terpenuhi (%)');plt.ylim(0,105);plt.legend();plt.tight_layout()
plt.savefig(OUT/'06_RWH_reliability_tank.png',dpi=170);plt.show()

In [ ]:
# 13. Metadata dan ringkasan status validasi; tidak mengklaim hasil faktual per rumah.
report = {
 'study_area':'Kalurahan Katongan, Nglipar, Gunungkidul',
 'rain_dataset':'CHIRPS v3 Daily RNL',
 'rain_period':'1996-01-01 through 2025-12-31',
 'rain_days':len(daily),
 'rainfall_annual_mean_mm':annual_mean,
 'building_count':len(b),
 'building_total_footprint_m2':float(b.area_in_meters.sum()),
 'rain_per_building_method':str(b.rain_data_method.iloc[0]),
 'base_rc':BASE_RUNOFF,
 'base_RCHP_m3yr_theoretical':float(b.RCHP_m3yr_RC80.sum()),
 'aoi_boundary_status':'Provided GEE administrative asset, source/version must remain documented',
 'building_status':'Model-detected footprint, NOT confirmed residence/roof area',
 'quality_data':'No lab observations: not evidence of potable quality',
 'tank_model':'Yield After Spillage (YAS), empty starting tank, no first flush or water-quality constraints',
 'risk_notes':[
  'CHIRPS 0.05 degree grid is coarse versus buildings; shared rain values do not indicate house-scale climate differences.',
  'Annual harvest volume is theoretical, not available/safe/potable water.',
  'Model uses scenario runoff, demand and tank volumes.',
  'No household occupancy, roof materials, tank structural feasibility or cost are verified.',
  'SPI tail results are sensitive with 30 calibration years; CDD sensitive to rainfall threshold.'
 ]
}
(OUT/'README_MODEL_METADATA.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps({k:v for k,v in report.items() if k not in ['risk_notes']},ensure_ascii=False,indent=2))

In [ ]:
# 14. ZIP siap diunduh untuk digabungkan ke WebGIS / laporan AGC.
zip_out = BASE/'AGC2027_Katongan_GeoA_GeoB_RWH_Integrated_Results.zip'
with zipfile.ZipFile(zip_out,'w',compression=zipfile.ZIP_DEFLATED) as zout:
    for f in sorted(OUT.iterdir()):
        zout.write(f,arcname=f.name)
print('ZIP siap:',zip_out,'(',round(zip_out.stat().st_size/1e6,2),'MB )')
if not os.environ.get('AGC_DISABLE_DOWNLOAD'):
    try:
        from google.colab import files
        files.download(str(zip_out))
    except ImportError:
        print('Unduh ZIP dari folder:',BASE)

## Lanjutan setelah notebook ini

1. Jalankan GEE sampling hujan piksel asli pada setiap centroid bangunan untuk memperoleh `Katongan_RWH_PerBuilding_SpatialGEE.csv`, lalu unggah berkas tersebut bersama ZIP dan jalankan ulang notebook. Karena grid CHIRPS kasar, beberapa bangunan akan menerima nilai hujan piksel yang sama; **itu benar**, bukan kekurangan visual.
2. Periksa footprint dengan sampel citra dan evaluasi salah deteksi, jenis bangunan, serta tapak yang melintasi batas Katongan.
3. Setelah ada unit penduduk/rumah tangga yang terverifikasi, baru hitung RCHP per kapita atau kebutuhan riil; jangan membagi seluruh potensi teoretis oleh populasi tanpa menjelaskan asumsi akses dan distribusinya.
4. Pertahankan modul optimasi titik bantuan eksisting sebagai pekerjaan **terpisah** dari simulasi distribusi hasil RWH.